In [ ]:
SEED = 15179996

In [ ]:
!pip install -q -U transformers datasets evaluate scikit-learn accelerate tqdm pandas huggingface_hub

In [ ]:
import torch
import pandas as pd
import numpy as np
import evaluate
from datasets import load_dataset, DatasetDict, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset Preparation
Using the same 1k test split from Sentiment140 as the LoRA study.

In [ ]:
def prepare_dataset(dataset_name="bdanko/sentiment140", test_size=1000):
    print(f"Loading dataset {dataset_name}...")
    dataset = load_dataset(dataset_name, split="train")
    df = dataset.to_pandas()

    # negative sentiment swapped to 1, original 4 is positive
    df['sentiment'] = df['sentiment'].replace(4, 1)

    # We only need the test set for the baseline
    # Using the same logic to ensure identical test set as the LORA study
    train_size = 5000
    train_df = df.sample(n=train_size, weights=df['sentiment'].map({0: 0.5, 1: 0.5}), random_state=SEED)
    remaining_df = df.drop(train_df.index)

    test_df_neg = remaining_df[remaining_df['sentiment'] == 0].sample(n=test_size // 2, random_state=SEED)
    test_df_pos = remaining_df[remaining_df['sentiment'] == 1].sample(n=test_size // 2, random_state=SEED)
    test_df = pd.concat([test_df_neg, test_df_pos]).sample(frac=1, random_state=SEED)

    return test_df.reset_index(drop=True)

test_df = prepare_dataset()
print(f"Test set size: {len(test_df)}")

# Raw Baseline: Mistral-7B-v0.3
Loading the base model for zero-shot evaluation.

In [ ]:
model_id = "mistralai/Mistral-7B-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# Zero-Shot Evaluation
We prompt the model and check the log-probabilities of 'Positive' vs 'Negative' tokens.

In [ ]:
pos_token = tokenizer.encode("Positive", add_special_tokens=False)[0]
neg_token = tokenizer.encode("Negative", add_special_tokens=False)[0]

print(f"Positive token ID: {pos_token} ('{tokenizer.decode([pos_token])}')")
print(f"Negative token ID: {neg_token} ('{tokenizer.decode([neg_token])}')")

def predict_sentiment(text):
    prompt = f"Sentiment classification. Text: {text}\nSentiment: "
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :] # Logits of the next token
        
        pos_logit = logits[pos_token].item()
        neg_logit = logits[neg_token].item()
        
        return 1 if pos_logit > neg_logit else 0

# Batch inference for speed
def batch_predict_sentiment(texts, batch_size=8):
    predictions = []
    prompts = [f"Sentiment classification. Text: {text}\nSentiment: " for text in texts]
    
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            # Get the logits for the last non-padding token in each sequence
            sequence_lengths = inputs.attention_mask.sum(dim=1) - 1
            last_token_logits = outputs.logits[torch.arange(outputs.logits.size(0)), sequence_lengths, :]
            
            pos_logits = last_token_logits[:, pos_token]
            neg_logits = last_token_logits[:, neg_token]
            
            batch_preds = (pos_logits > neg_logits).int().cpu().numpy().tolist()
            predictions.extend(batch_preds)
            
    return predictions

In [ ]:
y_true = test_df['sentiment'].values
y_pred = batch_predict_sentiment(test_df['text'].tolist(), batch_size=16)

# Metrics
Calculating Accuracy, Precision, Recall, and F1.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

results = {
    "accuracy": accuracy_metric.compute(predictions=y_pred, references=y_true)["accuracy"],
    "precision": precision_metric.compute(predictions=y_pred, references=y_true)["precision"],
    "recall": recall_metric.compute(predictions=y_pred, references=y_true)["recall"],
    "f1": f1_metric.compute(predictions=y_pred, references=y_true)["f1"],
}

print("\n--- Raw Baseline Metrics ---")
for k, v in results.items():
    print(f"{k.capitalize()}: {v:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - Raw Baseline')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()